# Create a SQL Server table from structural metadata

This tutorial demonstrates how pysdmx can use structural metadata to generate a SQL Server `CREATE TABLE` statement.

## Step 1: Connect to an SDMX-RESTv2 service containing the required structures

In [8]:
from pysdmx.api.fmr import RegistryClient
from pysdmx.toolkit.sqlsrv import create_table

client = RegistryClient("https://registry.sdmx.org/sdmx/v2")

## Step 2: Retrieve the required schema

In [9]:
schema = client.get_schema("datastructure", "IMF", "BOP", "2.6.0")

## Step 3: Inspect the schema

Notice the list of components, data type, facets, role and required fields.

In [12]:
for component in schema.components:
    print(component.id)

freq = schema.components["FREQ"]
print(f"Details about the {freq.id} ({freq.name}) {freq.role}")
print(f"  - Data type: {freq.dtype}")
print(f"  - Allowed codes: {', '.join([c.id for c in freq.enumeration])}")
print(f"  - Facets: {freq.facets}")

FREQ
ADJUSTMENT
REF_AREA
COUNTERPART_AREA
REF_SECTOR
COUNTERPART_SECTOR
FLOW_STOCK_ENTRY
ACCOUNTING_ENTRY
INT_ACC_ITEM
FUNCTIONAL_CAT
INSTR_ASSET
MATURITY
UNIT_MEASURE
CURRENCY_DENOM
VALUATION
COMP_METHOD
TYPE_ENTITY
TIME_PERIOD
OBS_VALUE
TIME_FORMAT
OBS_STATUS
CONF_STATUS
PRE_BREAK_VALUE
COMMENT_OBS
COMMENT_TS
TITLE
UNIT_MULT
DECIMALS
TIME_PER_COLLECT
REF_PERIOD_DETAIL
COMPILING_ORG
DATA_COMP
DISS_ORG
TITLE_COMPL
Details about the FREQ (Frequency) Dimension
  - Data type: String
  - Allowed codes: A, S, Q, M, W, D, H, B, N
  - Facets: min_length: 1, max_length: 1


## Step 4: Generate a SQL Server `CREATE TABLE` statement from the DSD

By default, the dimension values form the primary key and an index is created for each dimension. Facets determine column sizes and each component's `required` property determines whether the corresponding column is nullable. Names are used as documentation.

In [4]:
create_stmt = create_table(schema)
print(create_stmt)

CREATE TABLE dbo.BOP (
    FREQ CHAR(1) NOT NULL, -- Frequency
    ADJUSTMENT NVARCHAR(7) NOT NULL, -- Adjustment indicator
    REF_AREA NVARCHAR(MAX) NOT NULL, -- Reference country or area
    COUNTERPART_AREA NVARCHAR(MAX) NOT NULL, -- Counterpart Area
    REF_SECTOR NVARCHAR(8) NOT NULL, -- Reference sector
    COUNTERPART_SECTOR NVARCHAR(8) NOT NULL, -- Counterpart sector
    FLOW_STOCK_ENTRY NVARCHAR(3) NOT NULL, -- Flows and stocks indicator
    ACCOUNTING_ENTRY NVARCHAR(MAX) NOT NULL, -- Accounting entries
    INT_ACC_ITEM NVARCHAR(MAX) NOT NULL, -- International accounts item
    FUNCTIONAL_CAT NVARCHAR(MAX) NOT NULL, -- Functional category
    INSTR_ASSET NVARCHAR(MAX) NOT NULL, -- Instrument and assets classification
    MATURITY NVARCHAR(6) NOT NULL, -- Maturity
    UNIT_MEASURE NVARCHAR(MAX) NOT NULL, -- Unit of measure
    CURRENCY_DENOM NVARCHAR(MAX) NOT NULL, -- Currency of denomination
    VALUATION NVARCHAR(2) NOT NULL, -- Valuation
    COMP_METHOD CHAR(1) NOT NULL, --